# Mode publish notebook

This notebook expects a Mode SQL query named `BOM Capacity Raw`.
Its final output is `final_df`, which you can add to the report or publish as a local Python dataset in Mode.

In [ ]:
import pandas as pd

FINAL_COLUMN_ORDER = [
    "PATH",
    "PATH_WITHOUT_REVISION",
    "PART_NUMBER",
    "REVISION",
    "PRODUCT_NAME",
    "PRODUCTION_STATE",
    "INDENT_LEVEL",
    "UOM",
    "TRACKING",
    "IS_CONSUMABLE_STORABLE",
    "PARENT_BOM",
    "PARENT_REVISION",
    "TOP_LEVEL_BOM",
    "TOP_LEVEL_REVISION",
    "QUANTITY",
    "PROCUREMENT_INTENT",
    "ADJUSTED_QUANTITY",
    "total rolled up quantity",
    "ADJUSTED_PROCUREMENT_INTENT",
    "GLOBAL_ALTERNATE_PART_NUMBERS",
    "SUBSTITUTE_PART_NUMBERS",
    "Supply Plan and On-Hand Alternates",
    "Latest PO",
    "Latest Supplier",
    "Latest PO Creation Date",
    "Product",
    "Variant",
    "System",
    "Subsystem",
    "Gross Demand for BOM Line in past 8 weeks",
    "Net Total Demand for Part Number in past 8 weeks",
    "Current Week Total Gross Demand",
    "Current Week Net Demand",
    "Current Week Net Total Demand",
    "Current Week Realized Demand Consumption (each)",
    "Gross Demand for BOM Line in next 8 weeks",
    "Net Total Demand for Part Number in next 8 weeks",
    "Quantity Received, 8-Week Rolling Average (each)",
    "Quantity Received, Average Since Last Receipt (each)",
    "Current On-Hand Quantity (each)",
    "Current Receiving & Pre-IQC Quantity (each)",
    "On Hand Quantity In Parents",
    "in-transit quantity including alternates",
    "in-transit inventory value including alternates",
    "On Hand Delta to Current Week Demand (each)",
    "Current Quarantine Quantity (each)",
    "Current Week Realized Supply (each)",
    "Total Supply Plan, next 8 weeks (each)",
    "Average Supply Plan, next 8 weeks (each)",
    "Quantity Received, 8-Week Rolling Average (each) with alternates",
    "Quantity Received, Average Since Last Receipt (each) with alternates",
    "Current On-Hand Quantity (each) with alternates",
    "Current Receiving & Pre-IQC Quantity (each) with alternates",
    "Weeks of Stock",
    "Weeks of Stock with In Transit",
    "in transit weeks of stock",
    "In Transit Weeks of Stock Of System's Minimum Weeks of Stock Part",
    "Current On Hand Quantity Including alternates and parents",
    "Current On Hand Inventory Value Including alternates and parents",
    "on hand product sets including alternates",
    "receiving & pre-iqc product sets",
    "on hand product sets including alternates and parents",
    "on hand + in transit product sets",
    "Current Quarantine Quantity (each) with alternates",
    "Current Week Realized Supply (each) with alternates",
    "Total Supply Plan, next 8 weeks (each) with alternates",
    "Average Supply Plan, next 8 weeks (each) with alternates",
    "On Hand Inventory Status",
    "Supply Plan Status",
    "Avg Shipments Status",
]

EACH_COLUMNS = [
    "Current Week Realized Demand Consumption",
    "Quantity Received, 8-Week Rolling Average",
    "Quantity Received, Average Since Last Receipt",
    "Current On-Hand Quantity",
    "Current Receiving & Pre-IQC Quantity",
    "Current Quarantine Quantity",
    "Current Week Realized Supply",
    "Total Supply Plan, next 8 weeks",
    "Average Supply Plan, next 8 weeks",
]

ZERO_FILL_COLUMNS = [
    "Gross Demand for BOM Line in past 8 weeks",
    "Net Total Demand for Part Number in past 8 weeks",
    "Current Week Total Gross Demand",
    "Current Week Net Demand",
    "Current Week Net Total Demand",
    "Current Week Realized Demand Consumption (each)",
    "Gross Demand for BOM Line in next 8 weeks",
    "Net Total Demand for Part Number in next 8 weeks",
    "Quantity Received, 8-Week Rolling Average (each)",
    "Quantity Received, Average Since Last Receipt (each)",
    "Current On-Hand Quantity (each)",
    "Current Receiving & Pre-IQC Quantity (each)",
    "On Hand Quantity In Parents",
    "in-transit quantity including alternates",
    "in-transit inventory value including alternates",
    "On Hand Delta to Current Week Demand (each)",
    "Current Quarantine Quantity (each)",
    "Current Week Realized Supply (each)",
    "Total Supply Plan, next 8 weeks (each)",
    "Average Supply Plan, next 8 weeks (each)",
    "Quantity Received, 8-Week Rolling Average (each) with alternates",
    "Quantity Received, Average Since Last Receipt (each) with alternates",
    "Current On-Hand Quantity (each) with alternates",
    "Current Receiving & Pre-IQC Quantity (each) with alternates",
    "Weeks of Stock",
    "Weeks of Stock with In Transit",
    "in transit weeks of stock",
    "In Transit Weeks of Stock Of System's Minimum Weeks of Stock Part",
    "Current On Hand Quantity Including alternates and parents",
    "Current On Hand Inventory Value Including alternates and parents",
    "Current Quarantine Quantity (each) with alternates",
    "Current Week Realized Supply (each) with alternates",
    "Total Supply Plan, next 8 weeks (each) with alternates",
    "Average Supply Plan, next 8 weeks (each) with alternates",
]

ROLLUP_COLUMN = "total rolled up quantity"
PRODUCT_SETS_COLUMN = "on hand product sets including alternates"
RECEIVING_PRE_IQC_PRODUCT_SETS_COLUMN = "receiving & pre-iqc product sets"
PARENT_ON_HAND_COLUMN = "On Hand Quantity In Parents"
COMBINED_ON_HAND_COLUMN = "Current On Hand Quantity Including alternates and parents"
COMBINED_PRODUCT_SETS_COLUMN = "on hand product sets including alternates and parents"
ON_HAND_IN_TRANSIT_PRODUCT_SETS_COLUMN = "on hand + in transit product sets"

def demanded_top_level_boms(df):
    top_level_revision = (
        df["TOP_LEVEL_REVISION"].fillna("")
        if "TOP_LEVEL_REVISION" in df.columns
        else pd.Series("", index=df.index)
    )
    demanded_top_levels = pd.DataFrame(
        {
            "TOP_LEVEL_BOM": df["TOP_LEVEL_BOM"],
            "TOP_LEVEL_REVISION": top_level_revision,
        }
    ).dropna(subset=["TOP_LEVEL_BOM"])

    if demanded_top_levels.empty:
        return demanded_top_levels.drop_duplicates()

    child_mask = pd.Series(False, index=df.index)
    if "PARENT_BOM" in df.columns:
        child_mask |= df["PARENT_BOM"].fillna("").astype(str).str.strip().ne("")
    if "INDENT_LEVEL" in df.columns:
        child_mask |= pd.to_numeric(df["INDENT_LEVEL"], errors="coerce").fillna(0).gt(0)

    child_part_numbers = set(df.loc[child_mask, "PART_NUMBER"].dropna())
    if child_part_numbers:
        demanded_top_levels = demanded_top_levels[
            ~demanded_top_levels["TOP_LEVEL_BOM"].isin(child_part_numbers)
        ]

    return demanded_top_levels.drop_duplicates()

def non_top_level_assemblies(df):
    if "PART_NUMBER" not in df.columns:
        return set()

    child_mask = pd.Series(False, index=df.index)
    if "PARENT_BOM" in df.columns:
        child_mask |= df["PARENT_BOM"].fillna("").astype(str).str.strip().ne("")
    if "INDENT_LEVEL" in df.columns:
        child_mask |= pd.to_numeric(df["INDENT_LEVEL"], errors="coerce").fillna(0).gt(0)

    return set(df.loc[child_mask, "PART_NUMBER"].dropna())

try:
    raw_df = datasets["BOM Capacity Raw"].copy()
except Exception:
    raw_df = datasets[0].copy()

final_df = raw_df.copy()

if ROLLUP_COLUMN in final_df.columns:
    final_df[ROLLUP_COLUMN] = pd.to_numeric(final_df[ROLLUP_COLUMN], errors="coerce").astype("float64")
else:
    required_rollup_columns = {
        "PART_NUMBER",
        "TOP_LEVEL_BOM",
        "ADJUSTED_QUANTITY",
        "ADJUSTED_PROCUREMENT_INTENT",
    }
    if required_rollup_columns.issubset(final_df.columns):
        top_level_revision = (
            final_df["TOP_LEVEL_REVISION"].fillna("")
            if "TOP_LEVEL_REVISION" in final_df.columns
            else pd.Series("", index=final_df.index)
        )
        rollup_input = pd.DataFrame(
            {
                "PART_NUMBER": final_df["PART_NUMBER"],
                "TOP_LEVEL_BOM": final_df["TOP_LEVEL_BOM"],
                "TOP_LEVEL_REVISION": top_level_revision,
                "ADJUSTED_QUANTITY": pd.to_numeric(final_df["ADJUSTED_QUANTITY"], errors="coerce").fillna(0),
                "ADJUSTED_PROCUREMENT_INTENT": final_df["ADJUSTED_PROCUREMENT_INTENT"],
            }
        ).dropna(subset=["PART_NUMBER", "TOP_LEVEL_BOM"])
        rollup_input = rollup_input[rollup_input["ADJUSTED_PROCUREMENT_INTENT"].eq("zipline_buy")]

        if not rollup_input.empty:
            demanded_top_levels = demanded_top_level_boms(final_df)
            if not demanded_top_levels.empty:
                rollup_input = rollup_input.merge(
                    demanded_top_levels,
                    on=["TOP_LEVEL_BOM", "TOP_LEVEL_REVISION"],
                    how="inner",
                )

            if not rollup_input.empty and not demanded_top_levels.empty:
                top_level_totals = (
                    rollup_input.groupby(["PART_NUMBER", "TOP_LEVEL_BOM", "TOP_LEVEL_REVISION"], dropna=False)[
                        "ADJUSTED_QUANTITY"
                    ]
                    .sum()
                    .reset_index()
                )
                part_totals = top_level_totals.groupby("PART_NUMBER")["ADJUSTED_QUANTITY"].max().reset_index()
                part_totals = part_totals.rename(columns={"ADJUSTED_QUANTITY": ROLLUP_COLUMN})
                final_df = final_df.merge(part_totals[["PART_NUMBER", ROLLUP_COLUMN]], on="PART_NUMBER", how="left")

for column in EACH_COLUMNS:
    each_column = f"{column} (each)"
    final_df[each_column] = pd.to_numeric(final_df.get(column), errors="coerce")
    if column in final_df.columns:
        final_df = final_df.drop(columns=[column])

    alternate_column = f"{column} with alternates"
    alternate_each_column = f"{each_column} with alternates"
    final_df[alternate_each_column] = pd.to_numeric(final_df.get(alternate_column), errors="coerce")
    if alternate_column in final_df.columns:
        final_df = final_df.drop(columns=[alternate_column])

if PARENT_ON_HAND_COLUMN in final_df.columns:
    final_df[PARENT_ON_HAND_COLUMN] = pd.to_numeric(final_df[PARENT_ON_HAND_COLUMN], errors="coerce").fillna(0)
else:
    on_hand_column = "Current On-Hand Quantity (each)"
    if on_hand_column not in final_df.columns:
        on_hand_column = "Current On-Hand Quantity"
    usage_column = "QUANTITY" if "QUANTITY" in final_df.columns else "ADJUSTED_QUANTITY"
    required_parent_columns = {"PART_NUMBER", "PARENT_BOM", on_hand_column, usage_column}
    if required_parent_columns.issubset(final_df.columns):
        eligible_parents = non_top_level_assemblies(final_df)
        if eligible_parents:
            parent_on_hand = (
                pd.DataFrame(
                    {
                        "PART_NUMBER": final_df["PART_NUMBER"],
                        "_parent_on_hand": pd.to_numeric(final_df[on_hand_column], errors="coerce").fillna(0),
                    }
                )
                .dropna(subset=["PART_NUMBER"])
                .groupby("PART_NUMBER")["_parent_on_hand"]
                .max()
            )
            usage_quantity = pd.to_numeric(final_df[usage_column], errors="coerce").fillna(0)
            eligible_parent = final_df["PARENT_BOM"].isin(eligible_parents)
            parent_quantity = final_df["PARENT_BOM"].map(parent_on_hand).fillna(0)
            final_df[PARENT_ON_HAND_COLUMN] = 0.0
            final_df.loc[eligible_parent, PARENT_ON_HAND_COLUMN] = (
                parent_quantity.loc[eligible_parent] * usage_quantity.loc[eligible_parent]
            )
        else:
            final_df[PARENT_ON_HAND_COLUMN] = 0.0

if PRODUCT_SETS_COLUMN in final_df.columns:
    final_df[PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[PRODUCT_SETS_COLUMN], errors="coerce").astype("float64")
elif ROLLUP_COLUMN in final_df.columns:
    on_hand_column = "Current On-Hand Quantity (each) with alternates"
    if on_hand_column in final_df.columns:
        rollup_quantity = pd.to_numeric(final_df[ROLLUP_COLUMN], errors="coerce")
        on_hand_quantity = pd.to_numeric(final_df[on_hand_column], errors="coerce").fillna(0)
        valid_rollup = rollup_quantity.notna() & rollup_quantity.ne(0)
        final_df[PRODUCT_SETS_COLUMN] = pd.Series(pd.NA, index=final_df.index, dtype="Float64")
        final_df.loc[valid_rollup, PRODUCT_SETS_COLUMN] = (
            on_hand_quantity.loc[valid_rollup] / rollup_quantity.loc[valid_rollup]
        )
        final_df[PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[PRODUCT_SETS_COLUMN], errors="coerce")

if "Commodity" in final_df.columns:
    final_df = final_df.drop(columns=["Commodity"])

for column in FINAL_COLUMN_ORDER:
    if column not in final_df.columns:
        final_df[column] = None

for column in ZERO_FILL_COLUMNS:
    if column in final_df.columns:
        final_df[column] = pd.to_numeric(final_df[column], errors="coerce").fillna(0)

if ROLLUP_COLUMN in final_df.columns:
    final_df[ROLLUP_COLUMN] = pd.to_numeric(final_df[ROLLUP_COLUMN], errors="coerce").astype("float64")
if PRODUCT_SETS_COLUMN in final_df.columns:
    final_df[PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[PRODUCT_SETS_COLUMN], errors="coerce").astype("float64")
if RECEIVING_PRE_IQC_PRODUCT_SETS_COLUMN in final_df.columns:
    final_df[RECEIVING_PRE_IQC_PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[RECEIVING_PRE_IQC_PRODUCT_SETS_COLUMN], errors="coerce").astype("float64")
if PARENT_ON_HAND_COLUMN in final_df.columns:
    final_df[PARENT_ON_HAND_COLUMN] = pd.to_numeric(final_df[PARENT_ON_HAND_COLUMN], errors="coerce").fillna(0).astype("float64")
if COMBINED_ON_HAND_COLUMN in final_df.columns:
    final_df[COMBINED_ON_HAND_COLUMN] = pd.to_numeric(final_df[COMBINED_ON_HAND_COLUMN], errors="coerce").fillna(0).astype("float64")
if COMBINED_PRODUCT_SETS_COLUMN in final_df.columns:
    final_df[COMBINED_PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[COMBINED_PRODUCT_SETS_COLUMN], errors="coerce").astype("float64")
if ON_HAND_IN_TRANSIT_PRODUCT_SETS_COLUMN in final_df.columns:
    final_df[ON_HAND_IN_TRANSIT_PRODUCT_SETS_COLUMN] = pd.to_numeric(final_df[ON_HAND_IN_TRANSIT_PRODUCT_SETS_COLUMN], errors="coerce").astype("float64")

final_df = final_df.loc[:, FINAL_COLUMN_ORDER]


In [ ]:
final_df